# AG-02: Customer Support Ticket Triage — Multi-Agent System Exploration & Evaluation

This notebook provides a comprehensive walkthrough of the **Customer Support Ticket Triage** agentic workflow, built using **LangGraph**, specialized agents, tool integrations, and human-in-the-loop oversight.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.graph import triage_app
from src.tools.crm_tool import crm_lookup_tool
from src.tools.policy_retriever import policy_retriever_tool
from src.tools.action_tool import action_engine_tool
from src.evaluate import run_evaluation
print('Environment initialized successfully!')

## 1. Inspect Knowledge Base Policies and Vector Index
The system indexes 5 corporate policy documents covering refunds, security, logistics, SLAs, and billing.

In [ ]:
print(f'Indexed {len(policy_retriever_tool.chunks)} policy chunks across knowledge base files.')
query = 'What is the autonomous refund limit for damaged merchandise?'
results = policy_retriever_tool.retrieve(query, top_k=2)
for r in results:
    print(f"Policy: {r['policy_id']} | Title: {r['title']} | Score: {r['relevance_score']}")
    print(f"Excerpt: {r['excerpt'][:180]}...\n")

## 2. Customer CRM Exploration
Let us load the mock CRM database and examine customer distributions by tier and fraud risk.

In [ ]:
crm_path = project_root / 'data' / 'crm_database.json'
with open(crm_path, 'r', encoding='utf-8') as f:
    customers = json.load(f)['customers']

df_crm = pd.DataFrame(list(customers.values()))[['customer_id', 'name', 'tier', 'account_status', 'lifetime_value', 'fraud_risk_score']]
df_crm

## 3. End-to-End Execution of Sample Scenarios
We execute Scenario 1 (autonomous low-value refund) and Scenario 2 (consequential high-value hardware return).

In [ ]:
scenarios_path = project_root / 'data' / 'test_scenarios.json'
with open(scenarios_path, 'r', encoding='utf-8') as f:
    scenarios = json.load(f)

# Run Scenario 1
res1 = triage_app.invoke(scenarios[0], config={'configurable': {'thread_id': 'nb-1'}})
print('=== SCENARIO 1 (Autonomous) ===')
print(f"Category: {res1['category']} | Requires Approval: {res1['requires_human_approval']} | Action: {res1['executed_action']}")
print(f"Final Response Excerpt:\n{res1['final_response'][:250]}...\n")

# Run Scenario 2
res2 = triage_app.invoke(scenarios[1], config={'configurable': {'thread_id': 'nb-2'}})
print('=== SCENARIO 2 (Consequential / Human Gate) ===')
print(f"Category: {res2['category']} | Requires Approval: {res2['requires_human_approval']} | Reason: {res2['approval_reason']}")
print(f"Risk Score: {res2['risk_score']} | Supervisor Decision: {res2['human_decision']}")

## 4. Benchmark Evaluation Metrics (All 10 Scenarios)
Execute the full evaluation suite and display performance metrics.

In [ ]:
eval_results = run_evaluation()
pd.DataFrame([eval_results['metrics']]).T.rename(columns={0: 'Metric Value'})

## 5. Audit Trail Inspection
Inspect the immutable step-by-step telemetry trace from Scenario 1.

In [ ]:
trace_df = pd.DataFrame(res1['audit_trail'])[['step', 'agent', 'action', 'duration_ms', 'status']]
trace_df